In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False


BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

MAIN_DATA_BRANCH_PATH = DATA_DIR / "main_data_branch_12m.csv"
ROBUSTNESS_DATA_BRANCH_PATH = DATA_DIR / "robustness_data_branch_12m.csv"
MACRO_PATH = DATA_DIR / "macro_country_fiscal_year_final.csv"

OUTDIR = OUTPUT_DIR / "beta_12m"
OUTDIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [MAIN_DATA_BRANCH_PATH, ROBUSTNESS_DATA_BRANCH_PATH, MACRO_PATH]
missing_inputs = [str(path) for path in REQUIRED_INPUTS if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing input files. Place the required CSV files in the data/ directory:\n"
        + "\n".join(missing_inputs)
    )


ID_COL = "orbis_id"
YEAR_COL = "fiscal_year"
COUNTRY_COL = "country_iso"
CUTOFF_COL = "cutoff_date"
TARGET_COL = "future_beta_12m_w"

RAW_BETA_COL = "hist_beta_12m_w"
SHRINK_COL = "hist_beta_12m_w_shrink1"
BLUME_PRED_COL = "pred_beta_12m_blume_empirical"
BETA_LINEAR_PRED_COL = "pred_beta_12m_beta_linear"

TARGET_WEEKS_COL = "n_weeks_target_12m"
MIN_TARGET_WEEKS = 52

DEV_START_YEAR = 2015
DEV_END_YEAR = 2017
TEST_START_YEAR = 2018
TEST_END_YEAR = 2023

TEST_FIRM_FRAC = 0.20
RANDOM_SEED = 42


CORE_NUMERIC_FEATURES = [
    "log_assets",
    "roa",
    "roe",
    "ebit_margin",
    "ebitda_margin",
    "asset_turnover",
    "current_ratio",
    "working_capital_to_assets",
    "lt_debt_to_assets",
    "lt_debt_to_equity",
    "tangible_assets_share",
]

MACRO_FEATURES = [
    "inflation_yoy_cutoff",
    "policy_rate_cutoff",
    "gdp_growth_lag1",
]

BETA_FEATURES = [
    "hist_beta_12m_w",
    "hist_beta_36m_w",
    "hist_beta_60m_w",
    "hist_beta_12m_w_shrink1",
    "hist_beta_36m_w_shrink1",
    "hist_beta_60m_w_shrink1",
]

BETA_LINEAR_FEATURES = [
    "hist_beta_12m_w",
    "hist_beta_36m_w",
    "hist_beta_60m_w",
]

CATEGORICAL_FEATURES = ["country_iso", "nace4"]

FEATURE_VARIANTS = {
    "beta_only": BETA_FEATURES,
    "core": CORE_NUMERIC_FEATURES + CATEGORICAL_FEATURES,
    "core_plus_betas": CORE_NUMERIC_FEATURES + BETA_FEATURES + CATEGORICAL_FEATURES,
    "core_macro_betas": CORE_NUMERIC_FEATURES + MACRO_FEATURES + BETA_FEATURES + CATEGORICAL_FEATURES,
}


RIDGE_GRID = [{"alpha": a} for a in [0.01, 0.1, 1.0, 10.0, 100.0]]

ENET_GRID = [
    {"alpha": a, "l1_ratio": l}
    for a in [0.001, 0.01, 0.1, 1.0]
    for l in [0.1, 0.5, 0.9]
]

RF_GRID = [
    {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 5, "max_features": 0.5},
    {"n_estimators": 300, "max_depth": None, "min_samples_leaf": 10, "max_features": 0.5},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 5, "max_features": 0.5},
    {"n_estimators": 300, "max_depth": 10, "min_samples_leaf": 10, "max_features": 0.7},
    {"n_estimators": 500, "max_depth": None, "min_samples_leaf": 10, "max_features": 0.7},
]

GBR_GRID = [
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 5, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2, "min_samples_leaf": 5, "subsample": 0.8},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2, "min_samples_leaf": 5, "subsample": 0.8},
    {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 3, "min_samples_leaf": 10, "subsample": 0.8},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 10, "subsample": 0.8},
]

XGB_GRID = [
    {"n_estimators": 200, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.5, "colsample_bytree": 0.5, "min_child_weight": 5, "reg_lambda": 3.0},
    {"n_estimators": 300, "max_depth": 2, "learning_rate": 0.03, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 5, "reg_lambda": 3.0},
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 5, "reg_lambda": 5.0},
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 5, "reg_lambda": 5.0},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.7, "min_child_weight": 10, "reg_lambda": 5.0},
    {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 10, "reg_lambda": 10.0},
]

MLP_GRID = [
    {
        "hidden_layer_sizes": (32,),
        "alpha": 0.0001,
        "learning_rate_init": 0.001,
        "max_iter": 400,
        "early_stopping": True,
        "validation_fraction": 0.15,
    },
    {
        "hidden_layer_sizes": (64,),
        "alpha": 0.001,
        "learning_rate_init": 0.001,
        "max_iter": 500,
        "early_stopping": True,
        "validation_fraction": 0.15,
    },
    {
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.001,
        "learning_rate_init": 0.0005,
        "max_iter": 600,
        "early_stopping": True,
        "validation_fraction": 0.15,
    },
]

STRING_COLUMNS = [
    "company_name",
    "bvd_id",
    "isin",
    "ticker",
    "country_iso",
    "country",
    "consolidation_code",
    "quoted",
    "inactive",
    "original_currency_last",
    "symbol",
    "market_index_symbol",
    "hist_12m_invalid_reason",
    "target_12m_invalid_reason",
    "hist_36m_invalid_reason",
    "target_36m_invalid_reason",
    "hist_60m_invalid_reason",
    "target_60m_invalid_reason",
    "nace4",
]

REQUIRED_PANEL_COLUMNS = [
    ID_COL,
    YEAR_COL,
    COUNTRY_COL,
    CUTOFF_COL,
    TARGET_COL,
    RAW_BETA_COL,
    SHRINK_COL,
    TARGET_WEEKS_COL,
]

REQUIRED_MACRO_COLUMNS = [
    COUNTRY_COL,
    YEAR_COL,
    CUTOFF_COL,
    "inflation_yoy_cutoff",
    "policy_rate_cutoff",
    "gdp_growth_lag1",
]

RUN_TYPES = ["practical_time_oos", "strict_time_entity_oos"]
BENCHMARK_NAMES = [RAW_BETA_COL, SHRINK_COL, BLUME_PRED_COL, BETA_LINEAR_PRED_COL]

_AGG_DICT = {
    "mean_mae": ("mae", "mean"),
    "sd_mae": ("mae", lambda s: float(s.std(ddof=0))),
    "mean_rmse": ("rmse", "mean"),
    "sd_rmse": ("rmse", lambda s: float(s.std(ddof=0))),
    "mean_r2": ("r2", "mean"),
    "sd_r2": ("r2", lambda s: float(s.std(ddof=0))),
    "n_folds": ("fold_id", "nunique"),
}


def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_r2(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae": mae(y_true, y_pred),
        "rmse": rmse(y_true, y_pred),
        "r2": safe_r2(y_true, y_pred),
        "n_obs": int(len(y_true)),
    }


def aggregate_fold_scores(fold_df, group_keys):
    return fold_df.groupby(group_keys, as_index=False).agg(**_AGG_DICT)


def onehot():
    import sklearn

    major, minor = (int(x) for x in sklearn.__version__.split(".")[:2])
    if (major, minor) >= (1, 2):
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(feature_list, scale_numeric=True):
    numeric_pool = set(CORE_NUMERIC_FEATURES + MACRO_FEATURES + BETA_FEATURES)
    categorical_pool = set(CATEGORICAL_FEATURES)

    numeric = [c for c in feature_list if c in numeric_pool]
    categoricals = [c for c in feature_list if c in categorical_pool]

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric),
            ("cat", cat_pipe, categoricals),
        ],
        remainder="drop",
    )


def xy(df, feature_list):
    X = df[feature_list].copy()
    y = pd.to_numeric(df[TARGET_COL], errors="coerce")
    return X, y


def fit_linear(train_df, test_df, feature_list, model_name, params=None):
    params = params or {}
    X_train, y_train = xy(train_df, feature_list)
    X_test, y_test = xy(test_df, feature_list)

    pre = make_preprocessor(feature_list, scale_numeric=True)

    if model_name == "ols":
        model = LinearRegression()
    elif model_name == "ridge":
        model = Ridge(random_state=RANDOM_SEED, **params)
    elif model_name == "elastic_net":
        model = ElasticNet(random_state=RANDOM_SEED, max_iter=20000, **params)
    else:
        raise RuntimeError(f"Unknown linear model: {model_name}")

    pipe = Pipeline([("preprocessor", pre), ("model", model)])
    pipe.fit(X_train, y_train)
    return pipe.predict(X_test), y_test.to_numpy(), pipe


def fit_tree(train_df, test_df, feature_list, model_name, params):
    X_train, y_train = xy(train_df, feature_list)
    X_test, y_test = xy(test_df, feature_list)

    pre = make_preprocessor(feature_list, scale_numeric=False)

    if model_name == "gradient_boosting":
        model = GradientBoostingRegressor(random_state=RANDOM_SEED, **params)
    elif model_name == "random_forest":
        model = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1, **params)
    elif model_name == "xgboost":
        if not XGB_AVAILABLE:
            raise RuntimeError("XGBoost not available")
        model = XGBRegressor(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            objective="reg:squarederror",
            **params,
        )
    else:
        raise RuntimeError(f"Unknown tree model: {model_name}")

    pipe = Pipeline([("preprocessor", pre), ("model", model)])
    pipe.fit(X_train, y_train)
    return pipe.predict(X_test), y_test.to_numpy(), pipe


def fit_mlp(train_df, test_df, feature_list, params):
    X_train, y_train = xy(train_df, feature_list)
    X_test, y_test = xy(test_df, feature_list)

    pre = make_preprocessor(feature_list, scale_numeric=True)
    model = MLPRegressor(random_state=RANDOM_SEED, **params)

    pipe = Pipeline([("preprocessor", pre), ("model", model)])
    pipe.fit(X_train, y_train)
    return pipe.predict(X_test), y_test.to_numpy(), pipe


def fit_model(train_df, test_df, feature_list, model_name, params=None):
    params = params or {}

    if model_name in {"ols", "ridge", "elastic_net"}:
        return fit_linear(train_df, test_df, feature_list, model_name, params)

    if model_name in {"gradient_boosting", "random_forest", "xgboost"}:
        return fit_tree(train_df, test_df, feature_list, model_name, params)

    if model_name == "mlp_regressor":
        return fit_mlp(train_df, test_df, feature_list, params)

    raise RuntimeError(f"Unknown model_name: {model_name}")


def fit_blume_on(train_df):
    sub = train_df[[RAW_BETA_COL, TARGET_COL]].dropna().copy()
    if len(sub) < 30:
        raise RuntimeError("Too few observations for empirical Blume fit")
    X = sm.add_constant(sub[RAW_BETA_COL])
    model = sm.OLS(sub[TARGET_COL], X).fit()
    return (
        float(model.params["const"]),
        float(model.params[RAW_BETA_COL]),
        float(model.rsquared),
        int(len(sub)),
    )


def apply_blume(df, intercept, slope):
    return intercept + slope * pd.to_numeric(df[RAW_BETA_COL], errors="coerce")


def fit_beta_linear_on(train_df):
    missing = [c for c in BETA_LINEAR_FEATURES if c not in train_df.columns]
    if missing:
        raise RuntimeError(f"Missing beta linear features: {missing}")

    sub = train_df[BETA_LINEAR_FEATURES + [TARGET_COL]].dropna().copy()
    if len(sub) < 30:
        raise RuntimeError("Too few observations for beta linear benchmark")

    X = sm.add_constant(sub[BETA_LINEAR_FEATURES])
    model = sm.OLS(sub[TARGET_COL], X).fit()
    return model, float(model.rsquared), int(len(sub))


def apply_beta_linear(df, model):
    X = df[BETA_LINEAR_FEATURES].copy().apply(pd.to_numeric, errors="coerce")
    X = sm.add_constant(X, has_constant="add")
    return pd.Series(model.predict(X), index=df.index)


def load_panel(path, label):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in REQUIRED_PANEL_COLUMNS if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label}: missing columns {missing}")

    for c in STRING_COLUMNS:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip()
            df[c] = df[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})

    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")
    df[COUNTRY_COL] = df[COUNTRY_COL].astype("string").str.strip().str.upper()
    df[CUTOFF_COL] = pd.to_datetime(df[CUTOFF_COL], errors="coerce")
    df[TARGET_WEEKS_COL] = pd.to_numeric(df[TARGET_WEEKS_COL], errors="coerce")

    exempt = set(STRING_COLUMNS) | {CUTOFF_COL}
    for c in df.columns:
        if c not in exempt:
            df[c] = pd.to_numeric(df[c], errors="ignore")

    df = df.dropna(
        subset=[
            ID_COL,
            YEAR_COL,
            COUNTRY_COL,
            CUTOFF_COL,
            TARGET_COL,
            RAW_BETA_COL,
            SHRINK_COL,
            TARGET_WEEKS_COL,
        ]
    ).copy()

    if df.duplicated([ID_COL, YEAR_COL]).sum() > 0:
        raise RuntimeError(f"{label}: duplicate firm-year rows")

    df["dataset"] = label
    return df


def load_macro(path):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in REQUIRED_MACRO_COLUMNS if c not in df.columns]
    if missing:
        raise RuntimeError(f"macro: missing columns {missing}")

    df[COUNTRY_COL] = df[COUNTRY_COL].astype("string").str.strip().str.upper()
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")
    df[CUTOFF_COL] = pd.to_datetime(df[CUTOFF_COL], errors="coerce")

    for c in ["inflation_yoy_cutoff", "policy_rate_cutoff", "gdp_growth_lag1"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=REQUIRED_MACRO_COLUMNS).copy()

    if df.duplicated([COUNTRY_COL, YEAR_COL]).sum() > 0:
        raise RuntimeError("macro: duplicate country-year rows")

    return df


def merge_macro(panel_df, macro_df, label):
    merged = panel_df.merge(
        macro_df,
        on=[COUNTRY_COL, YEAR_COL],
        how="left",
        validate="many_to_one",
        suffixes=("", "_macro"),
    )

    if merged["inflation_yoy_cutoff"].isna().any():
        raise RuntimeError(f"{label}: missing macro rows after merge")

    macro_cutoff_col = f"{CUTOFF_COL}_macro"
    if macro_cutoff_col in merged.columns:
        mismatch = merged[merged[CUTOFF_COL] != merged[macro_cutoff_col]]
        if len(mismatch) > 0:
            raise RuntimeError(f"{label}: panel and macro cutoff_date mismatch")

    drop_cols = [c for c in [macro_cutoff_col, "macro_reference_month_end"] if c in merged.columns]
    if drop_cols:
        merged = merged.drop(columns=drop_cols)

    return merged


def apply_full_target_filter(df):
    return df[df[TARGET_WEEKS_COL] >= MIN_TARGET_WEEKS].copy()


def audit_panel(df_pre, df_post, dataset_name):
    rows = [
        {"dataset": dataset_name, "audit_item": "n_rows_pre_filter", "value": int(len(df_pre))},
        {"dataset": dataset_name, "audit_item": "n_rows_post_filter", "value": int(len(df_post))},
        {"dataset": dataset_name, "audit_item": "n_firms_pre_filter", "value": int(df_pre[ID_COL].nunique())},
        {"dataset": dataset_name, "audit_item": "n_firms_post_filter", "value": int(df_post[ID_COL].nunique())},
        {"dataset": dataset_name, "audit_item": "min_year_post_filter", "value": int(df_post[YEAR_COL].min())},
        {"dataset": dataset_name, "audit_item": "max_year_post_filter", "value": int(df_post[YEAR_COL].max())},
    ]

    cutoff_counts = df_post.groupby(YEAR_COL)[CUTOFF_COL].nunique().reset_index()
    bad_cutoff = cutoff_counts[cutoff_counts[CUTOFF_COL] != 1]
    rows.append(
        {
            "dataset": dataset_name,
            "audit_item": "years_with_nonunique_cutoff_date_post_filter",
            "value": int(len(bad_cutoff)),
        }
    )

    for c in [TARGET_COL, RAW_BETA_COL, SHRINK_COL]:
        rows.append(
            {
                "dataset": dataset_name,
                "audit_item": f"missing_{c}_post_filter",
                "value": int(df_post[c].isna().sum()),
            }
        )

    for y, g in df_post.groupby(YEAR_COL):
        rows.extend(
            [
                {
                    "dataset": dataset_name,
                    "audit_item": f"n_rows_year_{int(y)}_post_filter",
                    "value": int(len(g)),
                },
                {
                    "dataset": dataset_name,
                    "audit_item": f"n_firms_year_{int(y)}_post_filter",
                    "value": int(g[ID_COL].nunique()),
                },
                {
                    "dataset": dataset_name,
                    "audit_item": f"median_target_weeks_year_{int(y)}_post_filter",
                    "value": float(pd.to_numeric(g[TARGET_WEEKS_COL], errors="coerce").median()),
                },
            ]
        )

    return pd.DataFrame(rows)


def get_feature_list(df, variant_name):
    requested = FEATURE_VARIANTS[variant_name]
    available = [c for c in requested if c in df.columns]
    if not available:
        raise RuntimeError(f"{variant_name}: no available features")
    return available


def compile_model_specs():
    specs = [
        ("ols", "linear"),
        ("ridge", "linear"),
        ("elastic_net", "linear"),
        ("gradient_boosting", "tree"),
        ("random_forest", "tree"),
        ("mlp_regressor", "neural_net"),
    ]
    if XGB_AVAILABLE:
        specs.append(("xgboost", "tree"))
    return specs


def get_param_grid(model_name):
    if model_name == "ols":
        return [{}]
    if model_name == "ridge":
        return RIDGE_GRID
    if model_name == "elastic_net":
        return ENET_GRID
    if model_name == "gradient_boosting":
        return GBR_GRID
    if model_name == "random_forest":
        return RF_GRID
    if model_name == "mlp_regressor":
        return MLP_GRID
    if model_name == "xgboost":
        return XGB_GRID if XGB_AVAILABLE else []
    raise RuntimeError(f"Unknown model for grid: {model_name}")


def rank_models(df):
    return df.sort_values(
        ["mean_mae", "mean_rmse", "sd_mae", "sd_rmse", "model_name"],
        ascending=[True, True, True, True, True],
    ).reset_index(drop=True)


def firm_frame(dev_df):
    if "log_assets" in dev_df.columns:
        g = dev_df.groupby(ID_COL, as_index=False).agg(
            country_iso=(COUNTRY_COL, "first"),
            median_log_assets=("log_assets", "median"),
        )
        nonmissing = g["median_log_assets"].dropna()
        q = min(5, int(nonmissing.nunique()))
        if q >= 2:
            try:
                g["size_bucket"] = pd.qcut(
                    g["median_log_assets"],
                    q=q,
                    labels=[f"q{i+1}" for i in range(q)],
                    duplicates="drop",
                ).astype(str)
            except Exception:
                g["size_bucket"] = "all"
        else:
            g["size_bucket"] = "all"
    else:
        g = dev_df.groupby(ID_COL, as_index=False).agg(country_iso=(COUNTRY_COL, "first"))
        g["size_bucket"] = "all"

    g["country_iso"] = g["country_iso"].astype(str)
    g["size_bucket"] = g["size_bucket"].astype(str)
    g["strata"] = g["country_iso"] + "__" + g["size_bucket"]
    return g


def split_entity_firms(dev_df):
    frame = firm_frame(dev_df)
    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=TEST_FIRM_FRAC,
        random_state=RANDOM_SEED,
    )

    X_dummy = np.zeros((len(frame), 1))
    y_strata = frame["strata"].astype(str)

    try:
        _, test_idx = next(splitter.split(X_dummy, y_strata))
    except Exception:
        frame = frame.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
        n_test = max(1, int(round(len(frame) * TEST_FIRM_FRAC)))
        test_idx = np.arange(n_test)

    frame["entity_role"] = "train_firm"
    frame.loc[frame.index[test_idx], "entity_role"] = "test_firm"
    return frame


def build_expanding_folds_from_dev(dev_df):
    years = sorted(int(y) for y in dev_df[YEAR_COL].dropna().unique().tolist())
    if len(years) < 2:
        raise RuntimeError("Not enough development years for expanding validation")
    return [{"train_end_year": int(max(years[:i])), "valid_year": int(years[i])} for i in range(1, len(years))]


def _build_fold_splits(dev_pool_df, folds):
    fold_splits = []
    for i, fold in enumerate(folds, start=1):
        train_df = dev_pool_df[dev_pool_df[YEAR_COL] <= fold["train_end_year"]].copy()
        valid_df = dev_pool_df[dev_pool_df[YEAR_COL] == fold["valid_year"]].copy()
        if not train_df.empty and not valid_df.empty:
            fold_splits.append(
                {
                    "fold_id": i,
                    "train_end_year": fold["train_end_year"],
                    "valid_year": fold["valid_year"],
                    "train_df": train_df,
                    "valid_df": valid_df,
                }
            )
    return fold_splits


def get_practical_splits(panel_df):
    dev_df = panel_df[(panel_df[YEAR_COL] >= DEV_START_YEAR) & (panel_df[YEAR_COL] <= DEV_END_YEAR)].copy()
    test_df = panel_df[(panel_df[YEAR_COL] >= TEST_START_YEAR) & (panel_df[YEAR_COL] <= TEST_END_YEAR)].copy()

    if dev_df.empty or test_df.empty:
        raise RuntimeError("practical_time_oos: empty dev or test")

    fold_splits = _build_fold_splits(dev_df, build_expanding_folds_from_dev(dev_df))
    if not fold_splits:
        raise RuntimeError("practical_time_oos: no valid folds")

    return fold_splits, dev_df.copy(), test_df, None


def get_strict_splits(panel_df):
    dev_df = panel_df[(panel_df[YEAR_COL] >= DEV_START_YEAR) & (panel_df[YEAR_COL] <= DEV_END_YEAR)].copy()
    holdout_df = panel_df[(panel_df[YEAR_COL] >= TEST_START_YEAR) & (panel_df[YEAR_COL] <= TEST_END_YEAR)].copy()

    if dev_df.empty or holdout_df.empty:
        raise RuntimeError("strict_time_entity_oos: empty dev or holdout")

    manifest = split_entity_firms(dev_df)
    train_firms = set(manifest.loc[manifest["entity_role"] == "train_firm", ID_COL].tolist())
    test_firms = set(manifest.loc[manifest["entity_role"] == "test_firm", ID_COL].tolist())

    dev_train_firms = dev_df[dev_df[ID_COL].isin(train_firms)].copy()
    test_df = holdout_df[holdout_df[ID_COL].isin(test_firms)].copy()

    if test_df.empty:
        raise RuntimeError("strict_time_entity_oos: empty test")

    fold_splits = _build_fold_splits(dev_train_firms, build_expanding_folds_from_dev(dev_train_firms))
    if not fold_splits:
        raise RuntimeError("strict_time_entity_oos: no valid folds")

    return fold_splits, dev_train_firms.copy(), test_df, manifest


def pred_frame(df_ref, y_true, y_pred, dataset_name, run_type, feature_variant, model_name):
    out = df_ref[[ID_COL, YEAR_COL]].copy()
    out["dataset"] = dataset_name
    out["run_type"] = run_type
    out["feature_variant"] = feature_variant
    out["model_name"] = model_name
    out["y_true"] = y_true
    out["y_pred"] = y_pred
    return out


def benchmark_prediction_frame(df_ref, dataset_name, run_type, model_name, values):
    out = df_ref[[ID_COL, YEAR_COL]].copy()
    out["dataset"] = dataset_name
    out["run_type"] = run_type
    out["feature_variant"] = "benchmark"
    out["model_name"] = model_name
    out["y_true"] = pd.to_numeric(df_ref[TARGET_COL], errors="coerce").to_numpy()
    out["y_pred"] = pd.to_numeric(values, errors="coerce").to_numpy()
    return out


def evaluate_benchmark_on(df_ref, dataset_name, run_type, model_name, values):
    sub = pd.DataFrame(
        {
            "y_true": pd.to_numeric(df_ref[TARGET_COL], errors="coerce"),
            "y_pred": pd.to_numeric(values, errors="coerce"),
        }
    ).dropna()

    return {
        "dataset": dataset_name,
        "run_type": run_type,
        "phase": "test",
        "feature_variant": "benchmark",
        "model_family": "benchmark",
        "model_name": model_name,
        **metrics(sub["y_true"], sub["y_pred"]),
    }


def evaluate_benchmarks_on_folds(fold_splits, dataset_name, run_type):
    rows = []

    for fold in fold_splits:
        train_df, valid_df = fold["train_df"], fold["valid_df"]
        base = {
            "dataset": dataset_name,
            "run_type": run_type,
            "feature_variant": "benchmark",
            "fold_id": fold["fold_id"],
            "train_end_year": fold["train_end_year"],
            "valid_year": fold["valid_year"],
        }

        y_true = pd.to_numeric(valid_df[TARGET_COL], errors="coerce")

        for bench_col in [RAW_BETA_COL, SHRINK_COL]:
            bench_pred = pd.to_numeric(valid_df[bench_col], errors="coerce")
            sub = pd.DataFrame({"y": y_true, "p": bench_pred}).dropna()
            rows.append({**base, "model_name": bench_col, **metrics(sub["y"], sub["p"])})

        try:
            a, b, _, _ = fit_blume_on(train_df)
            blume_pred = apply_blume(valid_df, a, b)
            sub = pd.DataFrame({"y": y_true, "p": blume_pred}).dropna()
            rows.append({**base, "model_name": BLUME_PRED_COL, **metrics(sub["y"], sub["p"])})
        except RuntimeError:
            pass

        try:
            bl_model, _, _ = fit_beta_linear_on(train_df)
            bl_pred = apply_beta_linear(valid_df, bl_model)
            sub = pd.DataFrame({"y": y_true, "p": bl_pred}).dropna()
            rows.append({**base, "model_name": BETA_LINEAR_PRED_COL, **metrics(sub["y"], sub["p"])})
        except RuntimeError:
            pass

    return rows


def tune_model_over_folds(fold_splits, feature_list, model_name, param_grid):
    fold_rows = []

    for param_id, params in enumerate(param_grid, start=1):
        for fold in fold_splits:
            pred, y_true, _ = fit_model(fold["train_df"], fold["valid_df"], feature_list, model_name, params)
            fold_rows.append(
                {
                    "fold_id": fold["fold_id"],
                    "train_end_year": fold["train_end_year"],
                    "valid_year": fold["valid_year"],
                    "model_name": model_name,
                    "param_id": param_id,
                    "params": json.dumps(params, sort_keys=True),
                    **metrics(y_true, pred),
                }
            )

    tune_fold_df = pd.DataFrame(fold_rows)
    tune_summary_df = aggregate_fold_scores(tune_fold_df, ["model_name", "param_id", "params"])
    tune_summary_df = rank_models(tune_summary_df)

    best_params = json.loads(tune_summary_df.iloc[0]["params"])
    best_param_id = int(tune_summary_df.iloc[0]["param_id"])
    return best_params, best_param_id, tune_fold_df, tune_summary_df


def extract_winning_fold_rows(tune_fold_df, best_param_id, dataset_name, run_type, feature_variant):
    rows = tune_fold_df[tune_fold_df["param_id"] == best_param_id].copy()
    rows["dataset"] = dataset_name
    rows["run_type"] = run_type
    rows["feature_variant"] = feature_variant
    return rows[
        [
            "dataset",
            "run_type",
            "feature_variant",
            "fold_id",
            "train_end_year",
            "valid_year",
            "model_name",
            "mae",
            "rmse",
            "r2",
            "n_obs",
        ]
    ]


def build_common_sample_comparison(predictions_all_models_df, benchmark_predictions_df, test_all_models_df):
    rows = []

    ml_preds = predictions_all_models_df.copy()
    bench_preds = benchmark_predictions_df.copy()

    for (ds, rt, fv, mn), ml_g in ml_preds.groupby(["dataset", "run_type", "feature_variant", "model_name"]):
        ml_sub = ml_g[[ID_COL, YEAR_COL, "y_true", "y_pred"]].dropna()

        for bench_name in BENCHMARK_NAMES:
            bench_g = bench_preds[
                (bench_preds["dataset"] == ds)
                & (bench_preds["run_type"] == rt)
                & (bench_preds["model_name"] == bench_name)
            ][[ID_COL, YEAR_COL, "y_pred"]].dropna().rename(columns={"y_pred": "bench_pred"})

            common = ml_sub.merge(bench_g, on=[ID_COL, YEAR_COL], how="inner")
            if len(common) < 10:
                continue

            rows.append(
                {
                    "dataset": ds,
                    "run_type": rt,
                    "feature_variant": fv,
                    "ml_model": mn,
                    "benchmark": bench_name,
                    "n_common": len(common),
                    **{f"ml_{k}": v for k, v in metrics(common["y_true"], common["y_pred"]).items()},
                    **{f"bench_{k}": v for k, v in metrics(common["y_true"], common["bench_pred"]).items()},
                }
            )

    return pd.DataFrame(rows)


macro_df = load_macro(MACRO_PATH)

main_branch_raw = merge_macro(load_panel(MAIN_DATA_BRANCH_PATH, "main_data_branch_12m"), macro_df, "main_data_branch_12m")
robustness_branch_raw = merge_macro(load_panel(ROBUSTNESS_DATA_BRANCH_PATH, "robustness_data_branch_12m"), macro_df, "robustness_data_branch_12m")

main_branch_df = apply_full_target_filter(main_branch_raw)
robustness_branch_df = apply_full_target_filter(robustness_branch_raw)

datasets = {
    "main_data_branch_12m": main_branch_df,
    "robustness_data_branch_12m": robustness_branch_df,
}

raw_versions = {
    "main_data_branch_12m": main_branch_raw,
    "robustness_data_branch_12m": robustness_branch_raw,
}

model_specs = compile_model_specs()

audit_frames = []
split_manifest_frames = []
dataset_summary_rows = []
fold_definition_rows = []
tuning_fold_frames = []
tuning_summary_frames = []
validation_fold_rows = []
test_all_models_rows = []
predictions_all_models_frames = []
benchmark_prediction_frames = []
blume_coef_rows = []
beta_linear_coef_rows = []
selected_by_variant_rows = []
selected_global_rows = []
selected_test_frames = []
selected_vs_benchmark_rows = []

for dataset_name, panel_df in datasets.items():
    audit_frames.append(audit_panel(raw_versions[dataset_name], panel_df, dataset_name))

    practical_folds, practical_refit_df, practical_test_df, _ = get_practical_splits(panel_df)
    strict_folds, strict_refit_df, strict_test_df, strict_manifest = get_strict_splits(panel_df)

    strict_manifest["dataset"] = dataset_name
    split_manifest_frames.append(strict_manifest)

    split_map = {
        "practical_time_oos": {
            "folds": practical_folds,
            "refit_df": practical_refit_df,
            "test_df": practical_test_df,
        },
        "strict_time_entity_oos": {
            "folds": strict_folds,
            "refit_df": strict_refit_df,
            "test_df": strict_test_df,
        },
    }

    for run_type, split_obj in split_map.items():
        fold_years = [(f["train_end_year"], f["valid_year"]) for f in split_obj["folds"]]
        dataset_summary_rows.append(
            {
                "dataset": dataset_name,
                "run_type": run_type,
                "n_folds": len(split_obj["folds"]),
                "folds": "; ".join(f"train<={a}, valid={b}" for a, b in fold_years),
                "n_refit_rows": len(split_obj["refit_df"]),
                "n_test_rows": len(split_obj["test_df"]),
                "n_refit_firms": int(split_obj["refit_df"][ID_COL].nunique()),
                "n_test_firms": int(split_obj["test_df"][ID_COL].nunique()),
            }
        )

        for fold in split_obj["folds"]:
            fold_definition_rows.append(
                {
                    "dataset": dataset_name,
                    "run_type": run_type,
                    "fold_id": fold["fold_id"],
                    "train_end_year": fold["train_end_year"],
                    "valid_year": fold["valid_year"],
                    "n_train_rows": len(fold["train_df"]),
                    "n_valid_rows": len(fold["valid_df"]),
                    "n_train_firms": int(fold["train_df"][ID_COL].nunique()),
                    "n_valid_firms": int(fold["valid_df"][ID_COL].nunique()),
                }
            )

        bench_fold_rows = evaluate_benchmarks_on_folds(split_obj["folds"], dataset_name, run_type)
        validation_fold_rows.extend(bench_fold_rows)

        blume_a, blume_b, blume_r2, blume_n = fit_blume_on(split_obj["refit_df"])
        blume_coef_rows.append(
            {
                "dataset": dataset_name,
                "run_type": run_type,
                "intercept": blume_a,
                "slope": blume_b,
                "r2_refit": blume_r2,
                "n_refit_obs": blume_n,
            }
        )

        beta_linear_model, beta_linear_r2, beta_linear_n = fit_beta_linear_on(split_obj["refit_df"])
        beta_linear_coef_rows.append(
            {
                "dataset": dataset_name,
                "run_type": run_type,
                "intercept": float(beta_linear_model.params.get("const", np.nan)),
                "coef_hist_beta_12m_w": float(beta_linear_model.params.get("hist_beta_12m_w", np.nan)),
                "coef_hist_beta_36m_w": float(beta_linear_model.params.get("hist_beta_36m_w", np.nan)),
                "coef_hist_beta_60m_w": float(beta_linear_model.params.get("hist_beta_60m_w", np.nan)),
                "r2_refit": beta_linear_r2,
                "n_refit_obs": beta_linear_n,
            }
        )

        blume_test_pred = apply_blume(split_obj["test_df"], blume_a, blume_b)
        beta_linear_test_pred = apply_beta_linear(split_obj["test_df"], beta_linear_model)

        for benchmark_name, benchmark_values in [
            (RAW_BETA_COL, split_obj["test_df"][RAW_BETA_COL]),
            (SHRINK_COL, split_obj["test_df"][SHRINK_COL]),
            (BLUME_PRED_COL, blume_test_pred),
            (BETA_LINEAR_PRED_COL, beta_linear_test_pred),
        ]:
            test_all_models_rows.append(
                evaluate_benchmark_on(split_obj["test_df"], dataset_name, run_type, benchmark_name, benchmark_values)
            )
            benchmark_prediction_frames.append(
                benchmark_prediction_frame(split_obj["test_df"], dataset_name, run_type, benchmark_name, benchmark_values)
            )

        for variant_name in FEATURE_VARIANTS:
            feature_list = get_feature_list(panel_df, variant_name)

            for model_name, model_family in model_specs:
                param_grid = get_param_grid(model_name)
                if not param_grid:
                    continue

                best_params, best_param_id, tune_fold_df, tune_summary_df = tune_model_over_folds(
                    split_obj["folds"], feature_list, model_name, param_grid
                )

                tune_fold_df["dataset"] = dataset_name
                tune_fold_df["run_type"] = run_type
                tune_fold_df["feature_variant"] = variant_name
                tuning_fold_frames.append(tune_fold_df)

                tune_summary_df["dataset"] = dataset_name
                tune_summary_df["run_type"] = run_type
                tune_summary_df["feature_variant"] = variant_name
                tuning_summary_frames.append(tune_summary_df)

                winning_fold_rows = extract_winning_fold_rows(
                    tune_fold_df,
                    best_param_id,
                    dataset_name,
                    run_type,
                    variant_name,
                )
                validation_fold_rows.extend(winning_fold_rows.to_dict("records"))

                pred_test, y_test, _ = fit_model(
                    split_obj["refit_df"],
                    split_obj["test_df"],
                    feature_list,
                    model_name,
                    best_params,
                )

                test_all_models_rows.append(
                    {
                        "dataset": dataset_name,
                        "run_type": run_type,
                        "phase": "test",
                        "feature_variant": variant_name,
                        "model_family": model_family,
                        "model_name": model_name,
                        **metrics(y_test, pred_test),
                    }
                )

                predictions_all_models_frames.append(
                    pred_frame(
                        split_obj["test_df"],
                        y_test,
                        pred_test,
                        dataset_name,
                        run_type,
                        variant_name,
                        model_name,
                    )
                )

audit_df = pd.concat(audit_frames, ignore_index=True) if audit_frames else pd.DataFrame()
split_manifest_df = pd.concat(split_manifest_frames, ignore_index=True) if split_manifest_frames else pd.DataFrame()
dataset_summary_df = pd.DataFrame(dataset_summary_rows)
fold_definitions_df = pd.DataFrame(fold_definition_rows)
tuning_folds_df = pd.concat(tuning_fold_frames, ignore_index=True) if tuning_fold_frames else pd.DataFrame()
tuning_summary_df = pd.concat(tuning_summary_frames, ignore_index=True) if tuning_summary_frames else pd.DataFrame()
validation_folds_df = pd.DataFrame(validation_fold_rows)
test_all_models_df = pd.DataFrame(test_all_models_rows)
predictions_all_models_df = (
    pd.concat(predictions_all_models_frames, ignore_index=True)
    if predictions_all_models_frames
    else pd.DataFrame()
)
benchmark_predictions_df = (
    pd.concat(benchmark_prediction_frames, ignore_index=True)
    if benchmark_prediction_frames
    else pd.DataFrame()
)
blume_coefficients_df = pd.DataFrame(blume_coef_rows)
beta_linear_coefficients_df = pd.DataFrame(beta_linear_coef_rows)

validation_summary_df = aggregate_fold_scores(
    validation_folds_df,
    ["dataset", "run_type", "feature_variant", "model_name"],
)
validation_summary_df = rank_models(validation_summary_df)

for dataset_name in datasets:
    for run_type in RUN_TYPES:
        sub = validation_summary_df[
            (validation_summary_df["dataset"] == dataset_name)
            & (validation_summary_df["run_type"] == run_type)
        ].copy()

        for feature_variant in list(FEATURE_VARIANTS.keys()) + ["benchmark"]:
            by_variant = sub[sub["feature_variant"] == feature_variant].copy()
            if by_variant.empty:
                continue
            by_variant = rank_models(by_variant)
            selected_by_variant_rows.append(by_variant.iloc[0].to_dict())

        global_sub = rank_models(sub[sub["feature_variant"] != "benchmark"].copy())
        if global_sub.empty:
            raise RuntimeError(f"No ML validation summary for {dataset_name} {run_type}")
        selected_global_rows.append(global_sub.iloc[0].to_dict())

selected_by_variant_df = pd.DataFrame(selected_by_variant_rows).sort_values(
    ["dataset", "run_type", "feature_variant", "mean_mae"]
).reset_index(drop=True)

selected_global_df = pd.DataFrame(selected_global_rows).sort_values(
    ["dataset", "run_type", "mean_mae"]
).reset_index(drop=True)

best_oos_leaderboard = (
    test_all_models_df[test_all_models_df["feature_variant"] != "benchmark"]
    .sort_values(["dataset", "run_type", "mae"])
    .groupby(["dataset", "run_type"])
    .head(5)
    .reset_index(drop=True)
)

common_sample_df = build_common_sample_comparison(
    predictions_all_models_df,
    benchmark_predictions_df,
    test_all_models_df,
)

best_per_split = (
    test_all_models_df[test_all_models_df["feature_variant"] != "benchmark"]
    .sort_values("mae")
    .groupby(["dataset", "run_type"])
    .first()
    .reset_index()
)[["dataset", "run_type", "feature_variant", "model_name"]]

common_sample_best_df = common_sample_df.merge(
    best_per_split,
    left_on=["dataset", "run_type", "feature_variant", "ml_model"],
    right_on=["dataset", "run_type", "feature_variant", "model_name"],
    how="inner",
).drop(columns=["model_name"], errors="ignore")

for _, winner in selected_global_df.iterrows():
    ds, rt, fv, mn = winner["dataset"], winner["run_type"], winner["feature_variant"], winner["model_name"]

    selected_test = test_all_models_df[
        (test_all_models_df["dataset"] == ds)
        & (test_all_models_df["run_type"] == rt)
        & (test_all_models_df["feature_variant"] == fv)
        & (test_all_models_df["model_name"] == mn)
    ].copy()

    if selected_test.empty:
        raise RuntimeError(f"Missing selected test row for {ds} {rt} {fv} {mn}")

    selected_test_frames.append(selected_test)

    pred_sub = predictions_all_models_df[
        (predictions_all_models_df["dataset"] == ds)
        & (predictions_all_models_df["run_type"] == rt)
        & (predictions_all_models_df["feature_variant"] == fv)
        & (predictions_all_models_df["model_name"] == mn)
    ].copy()

    if pred_sub.empty:
        raise RuntimeError(f"Missing selected predictions for {ds} {rt} {fv} {mn}")

    ml_eval = pd.DataFrame(
        {
            "y_true": pd.to_numeric(pred_sub["y_true"], errors="coerce"),
            "y_pred": pd.to_numeric(pred_sub["y_pred"], errors="coerce"),
        }
    ).dropna()

    selected_vs_benchmark_rows.append(
        {
            "dataset": ds,
            "run_type": rt,
            "feature_variant": fv,
            "method": mn,
            **metrics(ml_eval["y_true"], ml_eval["y_pred"]),
        }
    )

    for benchmark_name in BENCHMARK_NAMES:
        bench_sub = benchmark_predictions_df[
            (benchmark_predictions_df["dataset"] == ds)
            & (benchmark_predictions_df["run_type"] == rt)
            & (benchmark_predictions_df["model_name"] == benchmark_name)
        ].copy()

        bench_eval = pd.DataFrame(
            {
                "y_true": pd.to_numeric(bench_sub["y_true"], errors="coerce"),
                "y_pred": pd.to_numeric(bench_sub["y_pred"], errors="coerce"),
            }
        ).dropna()

        selected_vs_benchmark_rows.append(
            {
                "dataset": ds,
                "run_type": rt,
                "feature_variant": fv,
                "method": benchmark_name,
                **metrics(bench_eval["y_true"], bench_eval["y_pred"]),
            }
        )

selected_test_results_df = (
    pd.concat(selected_test_frames, ignore_index=True)
    if selected_test_frames
    else pd.DataFrame()
)

selected_global_vs_benchmarks_df = pd.DataFrame(selected_vs_benchmark_rows).sort_values(
    ["dataset", "run_type", "mae", "method"]
).reset_index(drop=True)

selected_predictions_df = pd.DataFrame()
if not selected_global_df.empty:
    keep = selected_global_df[["dataset", "run_type", "feature_variant", "model_name"]].copy()
    selected_predictions_df = predictions_all_models_df.merge(
        keep,
        on=["dataset", "run_type", "feature_variant", "model_name"],
        how="inner",
        validate="many_to_one",
    )

audit_df.to_csv(OUTDIR / "audit_12m.csv", index=False)
dataset_summary_df.to_csv(OUTDIR / "dataset_summary_12m.csv", index=False)
fold_definitions_df.to_csv(OUTDIR / "fold_definitions_12m.csv", index=False)
split_manifest_df.to_csv(OUTDIR / "firm_split_manifest_12m.csv", index=False)
tuning_folds_df.to_csv(OUTDIR / "tuning_folds_12m.csv", index=False)
tuning_summary_df.to_csv(OUTDIR / "tuning_summary_12m.csv", index=False)
validation_folds_df.to_csv(OUTDIR / "validation_folds_12m.csv", index=False)
validation_summary_df.to_csv(OUTDIR / "validation_summary_12m.csv", index=False)
test_all_models_df.to_csv(OUTDIR / "test_all_models_12m.csv", index=False)
selected_by_variant_df.to_csv(OUTDIR / "selected_models_by_variant_12m.csv", index=False)
selected_global_df.to_csv(OUTDIR / "selected_global_models_12m.csv", index=False)
selected_test_results_df.to_csv(OUTDIR / "selected_test_results_12m.csv", index=False)
predictions_all_models_df.to_csv(OUTDIR / "predictions_all_models_12m.csv", index=False)
selected_predictions_df.to_csv(OUTDIR / "predictions_selected_global_12m.csv", index=False)
benchmark_predictions_df.to_csv(OUTDIR / "benchmark_predictions_12m.csv", index=False)
selected_global_vs_benchmarks_df.to_csv(OUTDIR / "selected_global_vs_benchmarks_12m.csv", index=False)
blume_coefficients_df.to_csv(OUTDIR / "empirical_blume_12m_coefficients.csv", index=False)
beta_linear_coefficients_df.to_csv(OUTDIR / "beta_linear_12m_coefficients.csv", index=False)
best_oos_leaderboard.to_csv(OUTDIR / "best_oos_leaderboard_12m.csv", index=False)
common_sample_df.to_csv(OUTDIR / "common_sample_comparison_12m.csv", index=False)
common_sample_best_df.to_csv(OUTDIR / "common_sample_best_12m.csv", index=False)

summary = {
    "target": TARGET_COL,
    "datasets": list(datasets.keys()),
    "run_types": RUN_TYPES,
    "feature_variants": list(FEATURE_VARIANTS.keys()),
    "benchmarks": BENCHMARK_NAMES,
    "development_window": [DEV_START_YEAR, DEV_END_YEAR],
    "test_window": [TEST_START_YEAR, TEST_END_YEAR],
    "target_window_filter": {
        "target_weeks_col": TARGET_WEEKS_COL,
        "min_target_weeks": MIN_TARGET_WEEKS,
    },
    "validation_design": "expanding_origin",
    "benchmark_symmetry": "benchmarks_evaluated_on_same_folds_as_ml_candidates",
    "validation_fold_source": "tuning_output_winning_param_id__no_extra_refit",
    "selected_global_scope": "ml_only__benchmarks_excluded_from_global_winner",
    "selection_rule": {
        "primary": "mean_mae",
        "secondary": "mean_rmse",
        "tie_breakers": ["sd_mae", "sd_rmse", "model_name"],
    },
    "outputs": {
        "audit": str(OUTDIR / "audit_12m.csv"),
        "dataset_summary": str(OUTDIR / "dataset_summary_12m.csv"),
        "fold_definitions": str(OUTDIR / "fold_definitions_12m.csv"),
        "split_manifest": str(OUTDIR / "firm_split_manifest_12m.csv"),
        "tuning_folds": str(OUTDIR / "tuning_folds_12m.csv"),
        "tuning_summary": str(OUTDIR / "tuning_summary_12m.csv"),
        "validation_folds": str(OUTDIR / "validation_folds_12m.csv"),
        "validation_summary": str(OUTDIR / "validation_summary_12m.csv"),
        "test_all_models": str(OUTDIR / "test_all_models_12m.csv"),
        "selected_by_variant": str(OUTDIR / "selected_models_by_variant_12m.csv"),
        "selected_global": str(OUTDIR / "selected_global_models_12m.csv"),
        "selected_test_results": str(OUTDIR / "selected_test_results_12m.csv"),
        "predictions_all_models": str(OUTDIR / "predictions_all_models_12m.csv"),
        "predictions_selected_global": str(OUTDIR / "predictions_selected_global_12m.csv"),
        "benchmark_predictions": str(OUTDIR / "benchmark_predictions_12m.csv"),
        "selected_global_vs_benchmarks": str(OUTDIR / "selected_global_vs_benchmarks_12m.csv"),
        "blume_coefficients": str(OUTDIR / "empirical_blume_12m_coefficients.csv"),
        "beta_linear_coefficients": str(OUTDIR / "beta_linear_12m_coefficients.csv"),
        "best_oos_leaderboard": str(OUTDIR / "best_oos_leaderboard_12m.csv"),
        "common_sample_comparison": str(OUTDIR / "common_sample_comparison_12m.csv"),
        "common_sample_best": str(OUTDIR / "common_sample_best_12m.csv"),
        "summary_json": str(OUTDIR / "summary_12m.json"),
    },
}

with open(OUTDIR / "summary_12m.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Completed. Output files:")
print("  best_oos_leaderboard_12m.csv  — top 5 modelů per split podle OOS MAE")
print("  common_sample_comparison_12m.csv — ML vs benchmarky na identickém průniku")
print("  common_sample_best_12m.csv — nejlepší ML model per split vs benchmarky na identickém průniku")
print()
print(json.dumps(summary, ensure_ascii=False, indent=2))